# Importing Christenson Transcription

We import Christensons direct transcription of the original Ximenez edition.

In [1]:
# quote_char = "’"

In [2]:
import pandas as pd
import numpy as np
from glob import glob 
import plotly_express as px
pd.set_option('display.max_columns', None)

In [3]:
OHCO = ['folio_num', 'side', 'lb_num', 'para_num']

Import into data frame.

In [4]:
ct_file_name =  'christenson-transcription-MOD.txt'
ct_text = open(ct_file_name, "r").readlines()
CT = pd.DataFrame({'line_str': ct_text})
CT.index.name = 'line_id'

In [5]:
CT.head()

,line_str
line_id,
0,Folio 1 recto\n
1,\n
2,ARE V XE OHER\n
3,Tzih varal Quiche vbi.\n
4,Varal xchicatzibah vi xchica-\n


Grab folio and side information.

In [6]:
CT.loc[CT.line_str.str.match("^Folio"), "folio_str"] = CT.line_str.str.split()
CT.folio_str = CT.folio_str.ffill()

In [7]:
CT = pd.concat([CT, CT.folio_str.apply(pd.Series)], axis=1)
CT = CT.drop(['folio_str', 0], axis=1)

In [8]:
CT.columns = ['line_str', 'folio_num', 'side']
CT.folio_num = CT.folio_num.astype(int)

Remove unwanted things.

In [9]:
CT = CT.loc[~CT.line_str.str.match(r"Folio \d")].copy()
# CT.line_str = CT.line_str.str.replace(r"\d+", "", regex=True)
CT = CT[~CT.line_str.str.match("^\s*\d+\s*$")]
CT = CT[~(CT.line_str == '')]
CT = CT[~CT.line_str.str.match(r"^\s*$")]

In [10]:
CT

,line_str,folio_num,side
line_id,,,
2,ARE V XE OHER\n,1,recto
3,Tzih varal Quiche vbi.\n,1,recto
4,Varal xchicatzibah vi xchica-\n,1,recto
5,"tiquiba vi oher tzih, vticaribal,\n",1,recto
6,"vxenabal puch ronohel xban,\n",1,recto
...,...,...,...
5697,qoheic quiche ri rumal mahabi\n,56,verso
5698,"chi ilbal re, qonabe oher cu-\n",56,verso
5699,mal ahauab zachinac chic. xere\n,56,verso


Add LB numbers.

In [11]:
CT['lb_num'] = 0
folio_num = ''
side = ''
lb_num = 0
for idx, row in CT.iterrows():
    if row['folio_num'] != folio_num:
        folio_num = row['folio_num']
    if row['side'] != side:
        side = row['side']
        lb_num = 0
    lb_num += 1
    CT.loc[idx, 'lb_num'] = lb_num    

Identify paragrapghs by indent.

In [12]:
para_pat = "^    " # 4 space indent
CT.loc[CT.line_str.str.match(para_pat), 'para_num'] = 1
CT.loc[CT.line_str.str.match(para_pat), 'para_num'] = np.array(range((CT.para_num == 1).sum())) + 1
CT.para_num = CT.para_num.ffill()
CT.para_num = CT.para_num.fillna(0)
CT.para_num = CT.para_num.astype(int)

Create OHCO index.

In [13]:
CT = CT.reset_index(drop=True).set_index(OHCO)

Remove editorial numbers.

In [14]:
CT.line_str = CT.line_str.str.replace(r"\d+(\w)", r"\1", regex=True).str.replace(r"(\w)\d+", r"\1", regex=True)

In [15]:
CT[CT.line_str.str.match(r"^\d")]

line_str
folio_num side  lb_num para_num                                     
56        recto 26     97          1  Ahau εalel vnabe ahau chuva-\n
                28     98        2  Ahau ah tzic vinac hun vnim ha\n
                29     98          3  Ahau εalel camha hun vnim ha\n
                30     98                4  Nima camha hun vnim ha\n
                31     98              5  Vchuch camha hun vnim ha\n
                32     98               6  Nima camha hun vnim ha.\n
                33     98         7  Nim chocoh nihaib hun vnim ha\n
                34     98              8  Ahau auilix hun vnim ha.\n
                35     98                9  Yacolatam hun vnim ha.\n

# Parse PARAs

In [16]:
PARA = CT.groupby(['para_num']).line_str.apply(" ".join).to_frame('doc_str')

Removed hyphens, etc.

In [17]:
PARA.doc_str = PARA.doc_str\
    .str.replace(r"-\n ", "", regex=True)\
    .str.replace(r"\n", "", regex=True)\
    .str.replace(r"\s+", " ", regex=True)\
    .str.strip()

In [18]:
PARA['n_chars'] = PARA.doc_str.str.len()

# Parse SENTs

In [19]:
SENT = PARA.doc_str.str.split(".")\
    .apply(pd.Series).stack().to_frame('sent_str')
SENT.sent_str = SENT.sent_str.str.strip()
SENT.index.names = ['para_num','sent_num']

In [20]:
# SENT.sample(10).sent_str.values

# Parse TOKENs

In [21]:
TOKEN = SENT.sent_str.str.split()\
    .apply(pd.Series).stack().to_frame('token_str')
TOKEN.index.names = ['para_num','sent_num','token_num']

In [22]:
TOKEN['term_str'] = TOKEN.token_str.str.lower()\
    .str.replace(",", "", regex=True)\
    .str.replace("[", "", regex=True)\
    .str.replace("(", "", regex=True)\
    .str.strip()

In [23]:
TOKEN

token_str term_str
para_num sent_num token_num                   
0        0        0               ARE      are
                  1                 V        v
                  2                XE       xe
                  3              OHER     oher
1        0        0              Tzih     tzih
...                               ...      ...
108      3        5           conohel  conohel
                  6            quiche   quiche
                  7               Sta      sta
         4        0              Cruz     cruz
                  1               vbi      vbi

[26904 rows x 2 columns]

# Extract VOCAB

In [24]:
VOCAB = TOKEN.term_str.value_counts().to_frame('n')
VOCAB.index.name = 'term_str'

In [25]:
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = np.log2(1/VOCAB.p)
VOCAB['h'] = VOCAB.p * VOCAB.i

In [26]:
VOCAB.head(20)

,n,p,i,h
term_str,,,,
ri,928,0.034493,4.857552,0.167552
ta,842,0.031296,4.997857,0.156415
cut,733,0.027245,5.197864,0.141616
qui,703,0.026130,5.258152,0.137395
chi,478,0.017767,5.814666,0.103308
vi,429,0.015946,5.970699,0.095206
xa,428,0.015908,5.974066,0.095038
hun,372,0.013827,6.176374,0.085400
are,350,0.013009,6.264322,0.081494


# Create BOW

In [27]:
BOW = TOKEN.groupby(["para_num", "term_str"]).term_str.count().unstack(fill_value=0)

In [28]:
BOW

term_str  0  1  2  3  4  5  6  7  8  9  a  abah  abanel  abanoh  abi  abix  \
para_num                                                                     
0         0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
1         0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
2         0  0  0  0  0  0  0  0  0  0  0     0       1       0    0     0   
3         0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
4         0  0  0  0  0  0  0  0  0  0  0     1       0       0    0     0   
...      .. .. .. .. .. .. .. .. .. .. ..   ...     ...     ...  ...   ...   
104       0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
105       0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
106       0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
107       0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   
108       0  0  0  0  0  0  0  0  0  0  0     0       0       0    0     0   

term_str  abixic  abixom  ac  aca  acaab  acab  acaba  acabal  acah  acahau  \
para_num                                                                      
0              0       0   0    0      0     0      0       0     0       0   
1              0       0   0    0      0     0      0       0     0       0   
2              0       0   0    0      0     0      0       0     0       0   
3              0       0   0    0      0     0      0       0     0       0   
4              0       0   0    0      0     0      0       0     0       0   
...          ...     ...  ..  ...    ...   ...    ...     ...   ...     ...   
104            0       0   0    0      0     0      0       0     0       0   
105            0       0   0    0      0     0      0       0     0       0   
106            0       0   0    0      0     0      0       0     0       0   
107            0       0   0    0      0     0      0       0     0       0   
108            0       0   0    0      0     0      0       0     0       0   

term_str  acalab  acam  acan  acanic  acanoc  acaroc  acarroc  acavitz  ach  \
para_num                                                                      
0              0     0     0       0       0       0        0        0    0   
1              0     0     0       0       0       0        0        0    0   
2              0     0     0       0       0       0        0        0    0   
3              0     0     0       0       0       0        0        0    0   
4              0     0     0       0       0       0        0        0    0   
...          ...   ...   ...     ...     ...     ...      ...      ...  ...   
104            0     0     0       0       0       0        0        0    0   
105            0     0     0       0       0       0        0        0    0   
106            0     0     0       0       0       0        0        0    0   
107            0     0     0       0       0       0        0        0    0   
108            0     0     0       0       0       0        0        0    0   

term_str  achac  achbilai  achbilan  achih  achihab  achihal  achihil  \
para_num                                                                
0             0         0         0      0        0        0        0   
1             0         0         0      0        0        0        0   
2             0         0         0      0        0        0        0   
3             0         0         0      0        0        0        0   
4             0         0         0      0        0        0        0   
...         ...       ...       ...    ...      ...      ...      ...   
104           0         0         0      0        0        0        0   
105           0         0         0      0        0        0        0   
106           0         0         0      0        0        0        0   
107           0         0         0      0        0        0        0   
108           0         0         0      0        0  

# Compute TFIDF

In [29]:
N_docs = len(BOW)
VOCAB['df'] = BOW.astype(bool).sum()
VOCAB['idf'] = np.log2(N_docs/VOCAB.df)
VOCAB['dfidf'] = VOCAB.df * VOCAB.idf

In [30]:
# VOCAB['dp'] = VOCAB.df / VOCAB.df.sum()
VOCAB['dp'] = VOCAB.df / N_docs
VOCAB['di'] = np.log2(1/VOCAB.dp)
VOCAB['dh'] = VOCAB.dp * VOCAB.di

In [31]:
TFIDF = BOW * VOCAB.idf

In [32]:
VOCAB['tfidf_mean'] = TFIDF.mean()

In [33]:
VOCAB.sort_values('dfidf', ascending=False).head(10).style.background_gradient(cmap="YlGnBu")

,n,p,i,h,df,idf,dfidf,dp,di,dh,tfidf_mean
term_str,,,,,,,,,,,
nim,62,0.002304,8.761337,0.020190,40,1.446256,57.850249,0.366972,1.446256,0.530736,0.822641
ahau,133,0.004944,7.660251,0.037868,40,1.446256,57.850249,0.366972,1.446256,0.530736,1.764698
mi,152,0.005650,7.467606,0.042190,41,1.410632,57.835925,0.376147,1.410632,0.530605,1.967120
vloc,110,0.004089,7.934173,0.032440,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.496386
quih,111,0.004126,7.921117,0.032681,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.509989
chiri,121,0.004497,7.796670,0.035065,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.646024
que,79,0.002936,8.411752,0.024700,42,1.375867,57.786410,0.385321,1.375867,0.530151,0.997188
chire,87,0.003234,8.272590,0.026751,42,1.375867,57.786410,0.385321,1.375867,0.530151,1.098169
chicut,95,0.003531,8.145677,0.028763,42,1.375867,57.786410,0.385321,1.375867,0.530151,1.199150


In [34]:
VOCAB[VOCAB.dh > .5].sort_values('dh', ascending=False)

,n,p,i,h,df,idf,dfidf,dp,di,dh,tfidf_mean
term_str,,,,,,,,,,,
nim,62,0.002304,8.761337,0.020190,40,1.446256,57.850249,0.366972,1.446256,0.530736,0.822641
ahau,133,0.004944,7.660251,0.037868,40,1.446256,57.850249,0.366972,1.446256,0.530736,1.764698
mi,152,0.005650,7.467606,0.042190,41,1.410632,57.835925,0.376147,1.410632,0.530605,1.967120
quih,111,0.004126,7.921117,0.032681,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.509989
chiri,121,0.004497,7.796670,0.035065,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.646024
vloc,110,0.004089,7.934173,0.032440,39,1.482782,57.828502,0.357798,1.482782,0.530537,1.496386
que,79,0.002936,8.411752,0.024700,42,1.375867,57.786410,0.385321,1.375867,0.530151,0.997188
curi,128,0.004758,7.715533,0.036708,42,1.375867,57.786410,0.385321,1.375867,0.530151,1.615697
chire,87,0.003234,8.272590,0.026751,42,1.375867,57.786410,0.385321,1.375867,0.530151,1.098169


In [35]:
VOCAB.loc[(VOCAB.dh > .3 ) & (VOCAB.tfidf_mean < 2.1)].sort_index()

,n,p,i,h,df,idf,dfidf,dp,di,dh,tfidf_mean
term_str,,,,,,,,,,,
a,49,0.001821,9.100823,0.016575,30,1.861294,55.838812,0.275229,1.861294,0.512283,0.836728
abah,27,0.001004,9.960646,0.009996,17,2.680721,45.572265,0.155963,2.680721,0.418094,0.664032
acab,59,0.002193,8.832890,0.019370,24,2.183222,52.397324,0.220183,2.183222,0.480709,1.181744
ah,95,0.003531,8.145677,0.028763,25,2.124328,53.108203,0.229358,2.124328,0.487231,1.851479
ahau,133,0.004944,7.660251,0.037868,40,1.446256,57.850249,0.366972,1.446256,0.530736,1.764698
...,...,...,...,...,...,...,...,...,...,...,...
ya,14,0.000520,10.908178,0.005676,10,3.446256,34.462562,0.091743,3.446256,0.316170,0.442638
zac,26,0.000966,10.015093,0.009679,19,2.520257,47.884879,0.174312,2.520257,0.439311,0.601162
zcaquin,19,0.000706,10.467606,0.007392,15,2.861294,42.919406,0.137615,2.861294,0.393756,0.498758


In [36]:
CHAR = pd.DataFrame([list(w) for w in VOCAB.index]).stack().value_counts().to_frame('n')

In [37]:
CHAR.sort_index()

,n
),1
-,10
0,1
1,1
2,2
3,1
4,1
5,1
6,1
7,1


# About $ε$

> The letter "ε" in the K'iche' language is pronounced by closing the throat and forcefully expelling air while keeping the tongue in the same position as for the letter "q". This produces a glottalized "q" sound.

Same as $q'$?

# Save

In [38]:
CT.to_csv("christenson-quiche-LINES.csv", index=True)

In [39]:
PARA.to_csv("christenson-quiche-PARA.csv", index=True)

In [40]:
TFIDF.to_csv("christenson-quiche-TFIDF.csv", index=True)